**Import tableau final**

In [1]:
!pip install --upgrade google-cloud-bigquery
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client(project="projet-riskhorizon-2276")


In [2]:
query = """
SELECT *
FROM `projet-riskhorizon-2276.3_Mart.tableau_final_table`
"""

df = client.query(query).to_dataframe()

df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,dette_totale_ext,ratio_dette_credit_ext,nb_credits_en_retard_ext,retard_max_ext,montant_total_retard_ext,nb_prolongations_ext,nb_types_credit_ext,nb_demandes_passees_int,nb_refus_passes_int,montant_total_emprunte_passe_int
0,355557,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4450.5,...,122050.98,0.474226,0,0,0.0,0,2,1,0,75312.0
1,211865,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4383.0,...,637623.00,0.421575,0,0,0.0,0,2,6,0,660501.0
2,364118,0,CASH_LOANS,F,False,True,0,29250.0,45000.0,4855.5,...,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,5,2,207724.5
3,143626,1,CASH_LOANS,F,False,True,0,31500.0,45000.0,5076.0,...,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,1,0,32503.5
4,372569,0,CASH_LOANS,F,False,False,0,31500.0,45000.0,4450.5,...,29817.00,0.047083,0,0,0.0,0,2,2,0,207616.5


In [3]:
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score


# --- 1. PRÉPARATION DES DONNÉES --
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

for col in categorical_features:
    X[col] = X[col].astype('category')

# Sauvegarde des noms de variables pour l'interprétation
feature_names = X.columns

# Split initial
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 2. CRÉATION DU PIPELINE CALIBRÉ ---

# a) On définit d'abord le modèle de base SANS class_weight="balanced"
base_model = HistGradientBoostingClassifier(
    max_iter=150,
    max_depth=4,
    min_samples_leaf=100,
    categorical_features='from_dtype',
    random_state=42
)

# b) On enveloppe ce modèle dans le calibrateur (méthode Isotonic ou Sigmoid)
# cv="prefit" ou un entier (ex: 3) permet de calibrer les probabilités
calibrated_model = CalibratedClassifierCV(estimator=base_model, method="sigmoid", cv=3)

# c) On intègre le tout dans votre pipeline de manière transparente
model_pipeline = Pipeline([
    ('classifier', calibrated_model)
])

# --- 3. VALIDATION CROISÉE ---

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')

print("VALIDATION CROISÉE (Gestion Native du Catégoriel)")
print(f"Scores AUC par fold : {cv_scores}")
print(f"Score AUC Moyen : {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print("-" * 30)

# --- 4. ENTRAÎNEMENT FINAL & ÉVALUATION ---

model_pipeline.fit(X_train, y_train)

y_pred_test = model_pipeline.predict_proba(X_test)[:, 1]
print(f"AUC Test Final : {roc_auc_score(y_test, y_pred_test):.4f}\n")


VALIDATION CROISÉE (Gestion Native du Catégoriel)
Scores AUC par fold : [0.7628282  0.76323843 0.76546067 0.76169983 0.76151368]
Score AUC Moyen : 0.7629 (+/- 0.0028)
------------------------------
AUC Test Final : 0.7618



In [4]:
# === SÉLECTION DU SEUIL OPTIMAL ===
#  Seuil  Accuracy   Recall  Précision
#   0.05  0.573545 0.803424   0.136423
#   0.08  0.704943 0.677543   0.168969
#   0.10  0.758537 0.601611   0.188347
#   0.12  0.795951 0.541591   0.207453
#   0.15  0.836862 0.448137   0.233768
#   0.20  0.873089 0.333938   0.269331
#   0.25  0.895236 0.239275   0.308251
#   0.30  0.907089 0.172004   0.347578
#   0.40  0.916683 0.090433   0.424787
#   0.50  0.918585 0.044512   0.456612

In [5]:
# --- 5. APPLICATION DU SEUIL BANCAIRE (0.08) & ANALYSE DE GENRE ---
SEUIL_BANQUE = 0.08
# Remplacement de .predict() par notre seuil personnalisé
y_pred_test_ajuste = (y_pred_test >= SEUIL_BANQUE).astype(int)

print(f"--- PERFORMANCES GLOBALES (SEUIL IMPOSE : {SEUIL_BANQUE}) ---")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_test_ajuste):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_test_ajuste):.4f} ")
print(f"Précision : {precision_score(y_test, y_pred_test_ajuste):.4f}\n")

# Masques pour l'équité des genres
masque_hommes = (df.loc[X_test.index, "CODE_GENDER"] == "M")
masque_femmes = (df.loc[X_test.index, "CODE_GENDER"] == "F")

def afficher_scores_par_genre(genre, masque):
    print(f"--- PERFORMANCES : {genre} ---")
    print(f"Nombre d'individus : {sum(masque)}")
    print(f"Recall    : {recall_score(y_test[masque], y_pred_test_ajuste[masque]):.4f}")
    print(f"Précision : {precision_score(y_test[masque], y_pred_test_ajuste[masque]):.4f}\n")

afficher_scores_par_genre("HOMMES (M)", masque_hommes)
afficher_scores_par_genre("FEMMES (F)", masque_femmes)

--- PERFORMANCES GLOBALES (SEUIL IMPOSE : 0.08) ---
Accuracy  : 0.7030
Recall    : 0.6890 
Précision : 0.1698

--- PERFORMANCES : HOMMES (M) ---
Nombre d'individus : 20931
Recall    : 0.7766
Précision : 0.1798

--- PERFORMANCES : FEMMES (F) ---
Nombre d'individus : 40569
Recall    : 0.6239
Précision : 0.1615



Proba par clients et selections des variables les plus influentes

In [6]:
import numpy as np
import pandas as pd
import shap

# --- 6. INTERPRÉTABILITÉ SHAP ---
X_full_df = X.copy()
for col in X_full_df.columns:
    if X_full_df[col].dtype.name == 'category':
        X_full_df[col] = X_full_df[col].cat.codes.astype(float)
    else:
        X_full_df[col] = X_full_df[col].astype(float)

# Correction technique : On extrait le modèle brut entraîné pour l'explainer
final_classifier = model_pipeline.named_steps['classifier'].calibrated_classifiers_[0].estimator
explainer = shap.TreeExplainer(final_classifier)
shap_values_full = explainer(X_full_df)
shap_full_df = pd.DataFrame(shap_values_full.values, columns=X_full_df.columns, index=df.index)

# Enregistrement des probabilités et décisions sur le df principal
df['proba_defaut'] = model_pipeline.predict_proba(X)[:, 1]
df['prediction_finale'] = (df['proba_defaut'] >= SEUIL_BANQUE).astype(int)

# Extraction du Top 3 SHAP
shap_abs = shap_full_df.abs()
top_3_indices = np.argsort(shap_abs.values, axis=1)[:, ::-1][:, :3]

for i in range(3):
    rang = i + 1
    df[f'variable_dominante_{rang}'] = shap_full_df.columns[top_3_indices[:, i]]
    df[f'impact_coef_{rang}'] = shap_full_df.values[np.arange(len(df)), top_3_indices[:, i]]
    df[f'sens_impact_{rang}'] = np.where(df[f'impact_coef_{rang}'] > 0, "Aggrave le risque", "Sécurise le dossier")

In [7]:
# --- 7. EXTRACTION DU DATAFRAME FINAL RESTREINT ---
colonnes_var_dominante = ['variable_dominante_1', 'variable_dominante_2', 'variable_dominante_3']
colonnes_uniques = df[colonnes_var_dominante].stack().unique().tolist()

# Construction de la liste des colonnes cibles
toutes_les_colonnes = ["SK_ID_CURR", "TARGET", "proba_defaut", "prediction_finale"]

for rang in range(1, 4):
    toutes_les_colonnes.extend([f'variable_dominante_{rang}', f'impact_coef_{rang}', f'sens_impact_{rang}'])

toutes_les_colonnes.extend(list(colonnes_uniques))
toutes_les_colonnes = list(dict.fromkeys(toutes_les_colonnes))

# Création du DataFrame d'export
df_final = df[toutes_les_colonnes].copy()

print("=== INSPECTION DU DATAFRAME DE SORTIE ===")
df_final.head()

=== INSPECTION DU DATAFRAME DE SORTIE ===


,SK_ID_CURR,TARGET,proba_defaut,prediction_finale,variable_dominante_1,impact_coef_1,sens_impact_1,variable_dominante_2,impact_coef_2,sens_impact_2,...,CNT_CHILDREN,nb_types_credit_ext,AMT_INCOME_TOTAL,RATIO_PRET,ORGANIZATION_TYPE,tx_endettement,Reste_a_vivre_par_personne,NAME_CONTRACT_TYPE,Reste_a_vivre,APPORT_ESTIME
0,355557,0,0.077565,0,EXT_SOURCE_2,0.982117,Aggrave le risque,NAME_EDUCATION_TYPE,-0.260786,Sécurise le dossier,...,0,2,27000.0,1.0,UNKNOWN,16.48,939.562,CASH_LOANS,1879.125,0.0
1,211865,0,0.012652,0,EXT_SOURCE_1,-0.498068,Sécurise le dossier,EXT_SOURCE_2,-0.334098,Sécurise le dossier,...,0,2,27000.0,1.0,UNKNOWN,16.23,942.375,CASH_LOANS,1884.750,0.0
2,364118,0,0.078308,0,EXT_SOURCE_2,0.427954,Aggrave le risque,NAME_EDUCATION_TYPE,-0.265457,Sécurise le dossier,...,0,<NA>,29250.0,1.0,UNKNOWN,16.60,1016.438,CASH_LOANS,2032.875,0.0
3,143626,1,0.117699,1,EXT_SOURCE_2,0.376412,Aggrave le risque,NAME_EDUCATION_TYPE,-0.218187,Sécurise le dossier,...,0,<NA>,31500.0,1.0,MEDICINE,16.11,1101.000,CASH_LOANS,2202.000,0.0
4,372569,0,0.009625,0,EXT_SOURCE_1,-0.572059,Sécurise le dossier,EXT_SOURCE_3,-0.357330,Sécurise le dossier,...,0,2,31500.0,1.0,UNKNOWN,14.13,2254.125,CASH_LOANS,2254.125,0.0


In [8]:
# --- 8. EXTRACTION DE L'IMPORTANCE GLOBALE (Moyenne & Pourcentage) ---

# 1. Calcul de l'importance absolue moyenne
global_importances = np.abs(shap_full_df).mean(axis=0)

# 2. Calcul du pourcentage de contribution relative
somme_importances = global_importances.sum()
pourcentages_importances = (global_importances / somme_importances) * 100

# 3. Création du DataFrame final d'importance
df_importance = pd.DataFrame({
    'Variable': global_importances.index,
    'Importance_SHAP_Moyenne': global_importances.values,
    'Contribution_%': pourcentages_importances.values
})

# 4. Tri par ordre décroissant
df_importance = df_importance.sort_values(by='Contribution_%', ascending=False).reset_index(drop=True)

print("=== IMPORTANCE GLOBALE DES VARIABLES AVEC POURCENTAGES ===")
# On affiche avec un formatage propre pour les décimales
print(df_importance.to_string(formatters={'Importance_SHAP_Moyenne': '{:,.4f}'.format, 'Contribution_%': '{:.2f}%'.format}))

=== IMPORTANCE GLOBALE DES VARIABLES AVEC POURCENTAGES ===
                            Variable Importance_SHAP_Moyenne Contribution_%
0                       EXT_SOURCE_2                  0.3251         10.83%
1                       EXT_SOURCE_3                  0.3168         10.55%
2                NAME_EDUCATION_TYPE                  0.2744          9.14%
3                       EXT_SOURCE_1                  0.1819          6.06%
4                    AMT_GOODS_PRICE                  0.1611          5.37%
5                        CODE_GENDER                  0.1387          4.62%
6                 NAME_FAMILY_STATUS                  0.1275          4.25%
7                         AMT_CREDIT                  0.1268          4.22%
8                    FLAG_OWN_REALTY                  0.1180          3.93%
9                     YEARS_EMPLOYED                  0.1063          3.54%
10                 ORGANIZATION_TYPE                  0.1043          3.47%
11            ratio_dette_cre

In [9]:
df_importance.head()

,Variable,Importance_SHAP_Moyenne,Contribution_%
0,EXT_SOURCE_2,0.325121,10.829772
1,EXT_SOURCE_3,0.316793,10.552368
2,NAME_EDUCATION_TYPE,0.274435,9.141402
3,EXT_SOURCE_1,0.181895,6.058928
4,AMT_GOODS_PRICE,0.161100,5.366233


In [10]:
import plotly.express as px

df_importance_top10 = df_importance.head(10)

fig_importance = px.bar(
    df_importance_top10,
    x='Contribution_%',
    y='Variable',
    orientation='h',
    title='Top 10 Variables par Pourcentage de Contribution SHAP (Plotly Express)',
    labels={'Contribution_%': 'Contribution (%)', 'Variable': 'Variable'},
    color='Contribution_%',
    color_continuous_scale=px.colors.sequential.Viridis
)

fig_importance.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_importance.show()

In [18]:
from google.cloud import bigquery

# Define your BigQuery project, dataset, and table name
project_id = "projet-riskhorizon-2276"
dataset_id = "3_Mart"
table_id = "coeff_HistGradientBoostingClassifier_variables"

destination_table = f"{project_id}.{dataset_id}.{table_id}"

# Export df_final to BigQueryS
df_importance.to_gbq(
    destination_table,
    project_id=project_id,
    if_exists='replace'
)

print(f"DataFrame 'df_final' successfully exported to BigQuery table: {destination_table}")

/tmp/ipykernel_47040/3968785746.py:11: FutureWarning:

to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq

100%|██████████| 1/1 [00:00<00:00, 4691.62it/s]

DataFrame 'df_final' successfully exported to BigQuery table: projet-riskhorizon-2276.3_Mart.coeff_HistGradientBoostingClassifier_variables


In [12]:
print(df_final['NAME_EDUCATION_TYPE'].unique())

['SECONDARY_/_SECONDARY_SPECIAL' 'HIGHER_EDUCATION' 'LOWER_SECONDARY'
 'INCOMPLETE_HIGHER' 'ACADEMIC_DEGREE']


In [13]:
# --- 9. NETTOYAGE / TRADUCTION ---


# --- ORGANIZATION_TYPE ---

mapping_secteurs = {
    'OTHER': 'Autres',
    'TRADE': 'Commerce et distribution',
    'CONSTRUCTION': 'Construction',
    'SCHOOL': 'Education',
    'UNKNOWN': 'Inconnu',
    'SELF-EMPLOYED': 'Indépendants / Artisans',
    'INDUSTRY': 'Industrie',
    'KINDERGARTEN': 'Petite enfance',
    'MEDICINE': 'Santé',
    'BUSINESS_ENTITY': 'Secteur privé',
    'GOVERNMENT': 'Secteur public',
    'SECURITY': 'Sécurité',
    'TRANSPORT': 'Transport'
}

# Remplacement des valeurs dans df_final
# On utilise .replace() car si une valeur n'est pas dans le dictionnaire, elle reste inchangée
df_final['ORGANIZATION_TYPE'] = df_final['ORGANIZATION_TYPE'].replace(mapping_secteurs)

print("=== VERIFICATION DES SECTEURS D'EMPLOI MODIFIÉS ===")
print(df_final['ORGANIZATION_TYPE'].unique())

# --- VARIABLE DOMINANTE ---

mapping_champs = {
    'EXT_SOURCE_2': 'score_2_ext',
    'EXT_SOURCE_3': 'score_3_ext',
    'NAME_EDUCATION_TYPE': 'niveau_scolaire',
    'EXT_SOURCE_1': 'score_1_ext',
    'AMT_GOODS_PRICE': 'valeur_bien',
    'CODE_GENDER': 'genre',
    'NAME_FAMILY_STATUS': 'statut_familial',
    'AMT_CREDIT': 'montant_credit',
    'FLAG_OWN_REALTY': 'proprietaire_logement',
    'YEARS_EMPLOYED': 'temps_emploi',
    'ORGANIZATION_TYPE': 'secteur_emploi',
    'RATIO_PRET': 'ratio_pret',
    'AMT_INCOME_TOTAL': 'revenus_annuel',
    'tx_endettement': 'taux_endettement',
    'montant_total_emprunte_passe_int': 'montant_total_emprunte_passe_int',
    'AGE': 'age',
    'CNT_CHILDREN': 'nb_enfants',
    'nb_refus_passes_int': 'nb_refus_credit_passes_int',
    'nb_demandes_passees_int': 'nb_demandes_credit_passees_int',
    'APPORT_ESTIME': 'apport_estime',
    'nb_credits_actifs_ext': 'nb_credit_actifs_ext',
    'FLAG_OWN_CAR': 'proprietaire_voiture',
    'NAME_CONTRACT_TYPE': 'type_emprunt',
    'total_credit_ext': 'montant_total_pret_ext',
    'Reste_a_vivre': 'reste_a_vivre',
    'dette_totale_ext': 'dette_totale_ext',
    'nb_credits_clotures_ext': 'nb_credit_clotures_ext',
    'nb_types_credit_ext': 'nb_types_credit_ext',
    'nb_credits_ext': 'nb_credits_ext',
    'retard_max_ext': 'nb_max_jours_retard_paiement_ext',
    'AMT_ANNUITY': 'montant_annuite',
    'Reste_a_vivre_par_personne': 'reste_a_vivre_par_personne',
    'NAME_INCOME_TYPE': 'source_revenus',
    'nb_prolongations_ext': 'nb_prolongations_ext',
    'montant_total_retard_ext': 'montant_total_retard_ext',
    'AVG_CREDIT_SCORE': 'moyenne_credit_score_ext',
    'nb_credits_en_retard_ext': 'nb_credit_en_retard_ext'
}

# Remplacement des valeurs dans df_final
# On utilise .replace() car si une valeur n'est pas dans le dictionnaire, elle reste inchangée
df_final['variable_dominante_1'] = df_final['variable_dominante_1'].replace(mapping_champs)
df_final['variable_dominante_2'] = df_final['variable_dominante_2'].replace(mapping_champs)
df_final['variable_dominante_3'] = df_final['variable_dominante_3'].replace(mapping_champs)

print("=== VERIFICATION DES CHAMPS MODIFIÉS ===")
print(df_final['variable_dominante_1'].unique())
print(df_final['variable_dominante_2'].unique())
print(df_final['variable_dominante_3'].unique())


# --- EDUCATION ---

mapping_education = {
   "ACADEMIC_DEGREE": "Doctorat (Bac+8)",
   "HIGHER_EDUCATION": "Enseignement supérieur (Diplômé)",
   "INCOMPLETE_HIGHER": "Supérieur incomplet / Étudiant",
   "SECONDARY_/_SECONDARY_SPECIAL": "Niveau Bac / Technique",
   "LOWER_SECONDARY": "Niveau Collège / Inférieur",
}


df_final["NAME_EDUCATION_TYPE"] = df_final["NAME_EDUCATION_TYPE"].replace(mapping_education)
# Sécurité pour le code manquant (souvent 'XNA' ou 'Unknown' dans ce dataset)
df["NAME_EDUCATION_TYPE"] = df["NAME_EDUCATION_TYPE"].fillna("INCONNU")

print("=== VERIFICATION EDUCATION ===")
print(df_final["NAME_EDUCATION_TYPE"].unique())

# --- FAMILY ---


mapping_statut_familial = {
   "MARRIED": "Marié",
   "WIDOW": "Veuf/Veuve",
   "SINGLE / NOT MARRIED": "Célibataire",
   "SEPARATED": "Séparé",
   "CIVIL MARRIAGE": "Concubinage / Union Civile"
}


df_final["NAME_FAMILY_STATUS"] = (df_final["NAME_FAMILY_STATUS"].map(mapping_statut_familial))
df_final["NAME_FAMILY_STATUS"] = df_final["NAME_FAMILY_STATUS"].fillna("INCONNU")

print("=== VERIFICATION FAMILY ===")
print(df_final["NAME_FAMILY_STATUS"].unique())

# --- TYPE CREDIT ---

traduction_contrats = {
    "CASH_LOANS": "Crédit amortissable",
    "REVOLVING_LOANS": "Crédit renouvelable",
}

df_final["NAME_CONTRACT_TYPE"] = (df_final["NAME_CONTRACT_TYPE"].map(traduction_contrats))
df_final["NAME_CONTRACT_TYPE"] = df_final["NAME_CONTRACT_TYPE"].fillna("CONTRAT_INCONNU")

print("=== VERIFICATION TYPE CREDIT ===")
print(df_final["NAME_CONTRACT_TYPE"].unique())


=== VERIFICATION DES SECTEURS D'EMPLOI MODIFIÉS ===
['Inconnu' 'Santé' 'Indépendants / Artisans' 'Autres' 'Industrie'
 'Transport' 'Secteur public' 'Secteur privé' 'Education'
 'Commerce et distribution' 'Construction' 'Petite enfance' 'Sécurité']
=== VERIFICATION DES CHAMPS MODIFIÉS ===
['score_2_ext' 'score_1_ext' 'score_3_ext' 'niveau_scolaire'
 'montant_total_emprunte_passe_int' 'temps_emploi' 'age' 'valeur_bien'
 'nb_refus_credit_passes_int' 'nb_credit_actifs_ext'
 'nb_credit_clotures_ext' 'genre' 'ratio_dette_credit_ext'
 'montant_total_retard_ext' 'ratio_pret' 'montant_total_pret_ext'
 'montant_credit' 'statut_familial' 'proprietaire_logement'
 'dette_totale_ext' 'type_emprunt' 'nb_types_credit_ext'
 'nb_demandes_credit_passees_int' 'taux_endettement']
['niveau_scolaire' 'score_2_ext' 'score_3_ext' 'score_1_ext' 'valeur_bien'
 'genre' 'nb_refus_credit_passes_int' 'ratio_dette_credit_ext'
 'temps_emploi' 'montant_total_pret_ext' 'montant_total_retard_ext' 'age'
 'nb_credit_actifs

In [14]:
df_final.head()

,SK_ID_CURR,TARGET,proba_defaut,prediction_finale,variable_dominante_1,impact_coef_1,sens_impact_1,variable_dominante_2,impact_coef_2,sens_impact_2,...,CNT_CHILDREN,nb_types_credit_ext,AMT_INCOME_TOTAL,RATIO_PRET,ORGANIZATION_TYPE,tx_endettement,Reste_a_vivre_par_personne,NAME_CONTRACT_TYPE,Reste_a_vivre,APPORT_ESTIME
0,355557,0,0.077565,0,score_2_ext,0.982117,Aggrave le risque,niveau_scolaire,-0.260786,Sécurise le dossier,...,0,2,27000.0,1.0,Inconnu,16.48,939.562,Crédit amortissable,1879.125,0.0
1,211865,0,0.012652,0,score_1_ext,-0.498068,Sécurise le dossier,score_2_ext,-0.334098,Sécurise le dossier,...,0,2,27000.0,1.0,Inconnu,16.23,942.375,Crédit amortissable,1884.750,0.0
2,364118,0,0.078308,0,score_2_ext,0.427954,Aggrave le risque,niveau_scolaire,-0.265457,Sécurise le dossier,...,0,<NA>,29250.0,1.0,Inconnu,16.60,1016.438,Crédit amortissable,2032.875,0.0
3,143626,1,0.117699,1,score_2_ext,0.376412,Aggrave le risque,niveau_scolaire,-0.218187,Sécurise le dossier,...,0,<NA>,31500.0,1.0,Santé,16.11,1101.000,Crédit amortissable,2202.000,0.0
4,372569,0,0.009625,0,score_1_ext,-0.572059,Sécurise le dossier,score_3_ext,-0.357330,Sécurise le dossier,...,0,2,31500.0,1.0,Inconnu,14.13,2254.125,Crédit amortissable,2254.125,0.0


***Récupération des colonnes supprimées***

In [15]:
# On récupère la liste de TOUTES les colonnes du DataFrame d'origine (df)
toutes_les_colonnes_origine = df.columns.tolist()

# n trouve les colonnes exclues (celles qui sont dans l'origine mais PAS dans df_final)
colonnes_non_selectionnees = [col for col in toutes_les_colonnes_origine if col not in df_final.columns]

# --- AFFICHAGE DU BILAN ---
print("=== ANALYSE DES COLONNES SUPPRIMÉES ===")
print(f"Nombre de colonnes supprimées : {len(colonnes_non_selectionnees)}")
print("-" * 40)

# Affichage de la liste des colonnes jetées (par blocs de 5 pour que ce soit lisible)
for i in range(0, len(colonnes_non_selectionnees), 5):
    print(colonnes_non_selectionnees[i:i+5])

print("-" * 40)

# CRÉATION DU DATAFRAME DES COLONNES EXCLUES
# On conserve souvent 'ref_client' comme clé de jointure au cas où
colonnes_a_garder_exclues = ["SK_ID_CURR"] + colonnes_non_selectionnees
df_exclues = df[colonnes_a_garder_exclues].copy()

print(f"Dimensions du DataFrame des colonnes exclues (df_exclues) : {df_exclues.shape}")

=== ANALYSE DES COLONNES SUPPRIMÉES ===
Nombre de colonnes supprimées : 8
----------------------------------------
['FLAG_OWN_CAR', 'AMT_ANNUITY', 'NAME_INCOME_TYPE', 'AVG_CREDIT_SCORE', 'nb_credits_ext']
['nb_credits_en_retard_ext', 'retard_max_ext', 'nb_prolongations_ext']
----------------------------------------
Dimensions du DataFrame des colonnes exclues (df_exclues) : (307497, 9)


In [16]:
from google.cloud import bigquery

# Define your BigQuery project, dataset, and table name
project_id = "projet-riskhorizon-2276"
dataset_id = "3_Mart"
table_id = "tableau_final_HistGradientBoostingClassifier_v2"

destination_table = f"{project_id}.{dataset_id}.{table_id}"

# Export df_final to BigQueryS
df_final.to_gbq(
    destination_table,
    project_id=project_id,
    if_exists='replace'
)

print(f"DataFrame 'df_final' successfully exported to BigQuery table: {destination_table}")

/tmp/ipykernel_47040/242533498.py:11: FutureWarning:

to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq

100%|██████████| 1/1 [00:00<00:00, 6374.32it/s]

DataFrame 'df_final' successfully exported to BigQuery table: projet-riskhorizon-2276.3_Mart.tableau_final_HistGradientBoostingClassifier_v2


**Etape 10 : Matrice de confusion**

In [17]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import plotly.express as px

# conversion des probabilités en classes (Seuil par défaut à 0.5)
y_pred_class = (y_pred_test >= 0.08).astype(int)

# calcul de la matrice brute et normalisée
cm = confusion_matrix(y_test, y_pred_class)
cm_normalized = confusion_matrix(y_test, y_pred_class, normalize='true') # Taux par ligne

# création du graphique Plotly
labels_etats = ['Non-Défaut (0)', 'Défaut (1)']

fig_cm = px.imshow(
    cm_normalized,
    text_auto=".2%", # Affiche les taux en pourcentage
    x=labels_etats,
    y=labels_etats,
    labels=dict(x="État PRÉDIT par le modèle", y="État RÉEL du client", color="Taux"),
    title="Matrice de Confusion (Transition Réel vs Prédit)",
    color_continuous_scale="Blues"
)

fig_cm.show()